# Model Profiling

In [ ]:
import sys
from pathlib import Path
# sys.path.insert(0, str(Path().resolve().parent)) 

In [ ]:
# parent of notebooks/ is the project root
project_root = Path().resolve().parent   
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import torch
from thop import profile, clever_format
from src.lightning_module.predictor import SetAnomalyPredictor
# import matplotlib.pyplot as plt

In [ ]:
# Import matplotlib with Agg backend to ensure saving works
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
model = SetAnomalyPredictor(
    input_dim=128,
    model_dim=128,
    num_classes=2,
    num_heads=8,
    num_layers=2,
    lr=1e-3,
)

In [ ]:
model.eval()

In [ ]:
# Profile for a set of size 16
x = torch.randn(1, 16, 128)
flops, params = profile(model, inputs=(x,), verbose=False)
flops, params = clever_format([flops, params], "%.3f")

In [ ]:
print(f"Parameters: {params}")
print(f"FLOPs per forward pass (set size 16): {flops}")

In [ ]:
# Show how FLOPs scale with set size
sizes = [4, 8, 16, 32, 64]
flops_list = []
for s in sizes:
    x = torch.randn(1, s, 128)
    f, _ = profile(model, inputs=(x,), verbose=False)
    flops_list.append(f)

In [ ]:
# Plot

# Create plot
plt.figure(figsize=(8, 5))
plt.plot(sizes, flops_list, marker='o', linestyle='-', linewidth=2, markersize=8)
plt.xlabel("Set Size (Sequence Length)")
plt.ylabel("FLOPs")
plt.title("Computational Cost of SetAnomalyPredictor")
plt.grid(True)
plt.tight_layout()  # prevents clipping

In [ ]:
# Save figure to project_root/results/
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)
save_path = results_dir / "flops_vs_set_size.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor='white')
print(f"Plot saved to {save_path}")

In [ ]:
from IPython.display import Image
display(Image(save_path))